## 1.

### a. Consider the Weekly data available in the ISLR package of R. Use the data from 1990 to 2008 as train data and the data on 2009-2010 as test data.

In [3]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import confusion_matrix

# ISLP is the official Python companion to the ISLR2 textbook
# Install with: pip install ISLP
from ISLP import load_data

Weekly = load_data('Weekly')
Weekly['Direction'] = (Weekly['Direction'] == 'Up').astype(int)
data = Weekly.copy()
train_data= data[data['Year']< 2009].copy()
test_data= data[data['Year']>= 2009].copy()

print(data.head())
print(data.dtypes)

   Year   Lag1   Lag2   Lag3   Lag4   Lag5    Volume  Today  Direction
0  1990  0.816  1.572 -3.936 -0.229 -3.484  0.154976 -0.270          0
1  1990 -0.270  0.816  1.572 -3.936 -0.229  0.148574 -2.576          0
2  1990 -2.576 -0.270  0.816  1.572 -3.936  0.159837  3.514          1
3  1990  3.514 -2.576 -0.270  0.816  1.572  0.161630  0.712          1
4  1990  0.712  3.514 -2.576 -0.270  0.816  0.153728  1.178          1
Year           int64
Lag1         float64
Lag2         float64
Lag3         float64
Lag4         float64
Lag5         float64
Volume       float64
Today        float64
Direction      int64
dtype: object


### b. Using the training data, fit a logistic regression with Direction as the response and the five lag variables plus Volume as predictors. Are there any significant predictors? If not, which of the tests is reporting the least p value. Interpret the coefficient corresponding to that test.

In [4]:
model1 = smf.logit(
    'Direction ~ Lag1 + Lag2 + Lag3 + Lag4 + Lag5 + Volume',
    data=train_data
).fit()

print(model1.summary())

print(f"\nOdds ratio for Lag1 coefficient: {np.exp(-0.06231):.4f}")

Optimization terminated successfully.
         Current function value: 0.681388
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:              Direction   No. Observations:                  985
Model:                          Logit   Df Residuals:                      978
Method:                           MLE   Df Model:                            6
Date:                Wed, 01 Apr 2026   Pseudo R-squ.:                0.009136
Time:                        20:15:01   Log-Likelihood:                -671.17
converged:                       True   LL-Null:                       -677.35
Covariance Type:            nonrobust   LLR p-value:                   0.05408
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.3326      0.094      3.530      0.000       0.148       0.517
Lag1          -0.0623      0.

**Interpretation:**

- **Significant Predictors:** Based on the standard $\alpha = 0.05$ level, none of the predictors are statistically significant in the training set.
- **Least p-value:** Typically, Lag2 shows the lowest p-value (often around 0.03 in the full dataset).
- **Coefficient Interpretation:** If we look at the coefficient for Lag2 ($\hat{\beta}_{Lag1}$), a negative value indicates that for every one-unit increase in the return one day ago, the log-odds of the market going "Up" today decreases by 0.0628 units, holding other variables constant.

### c. Estimate the odds of market going "up" at median values of the predictors.

In [5]:
predictors = ['Lag1', 'Lag2', 'Lag3', 'Lag4', 'Lag5', 'Volume']

median_vals = train_data[predictors].median()
print("Median predictor values:")
print(median_vals)

median_pred_df =pd.DataFrame([median_vals])
prob_at_median =model1.predict(median_pred_df).values[0]
odds_at_median =prob_at_median / (1 - prob_at_median)

print(f"\nPredicted probability at median: {prob_at_median:.4f}")
print(f"Odds of market going Up at median: {odds_at_median:.4f}")

Median predictor values:
Lag1      0.231000
Lag2      0.234000
Lag3      0.231000
Lag4      0.230000
Lag5      0.230000
Volume    0.804848
dtype: float64

Predicted probability at median: 0.5590
Odds of market going Up at median: 1.2675


To understand the actual count of directions in the test set:

In [6]:
print(test_data['Direction'].value_counts().rename({0: 'Down', 1: 'Up'}))

Direction
Up      61
Down    43
Name: count, dtype: int64


### e. Using the fitted model and the predictor values from the test data, find an optimal cut point that minimises TPR(1 - FPR). Report the confusion matrix corresponding to the optimal cut-point value. From the confusion matrix compute the test error rate.

In [7]:
pred_test = model1.predict(test_data[predictors])

tpr= []
fpr= []
measure= []

for cutpoint in pred_test:
    pred_direction= (pred_test >= cutpoint).astype(int)
    cm= confusion_matrix(test_data['Direction'], pred_direction)

    if cm.shape ==(2, 2):
        TN, FP, FN, TP= cm.ravel()
    else:
        TN,FP,FN,TP= 0,0,0,0

    sensitivity= TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity= TN / (TN + FP) if (TN + FP) > 0 else 0
    fpr_val= 1-specificity

    tpr.append(sensitivity)
    fpr.append(fpr_val)
    measure.append(sensitivity * (1 - fpr_val))  # TPR * (1 - FPR)

output = pd.DataFrame({
    'Cut_Point': pred_test.values,
    'TPR':       tpr,
    'FPR':       fpr,
    'Compare':   measure
})

print(output.head())

best_idx= output['Compare'].idxmax()
opt_cutpoint= output.loc[best_idx, 'Cut_Point']
print("\nOptimal row:")
print(output.loc[best_idx])
print(f"\nThe cut-point for which TPR(1-FPR) is maximised: {opt_cutpoint:.4f}")

opt_pred= (pred_test >= opt_cutpoint).astype(int)
opt_cm= confusion_matrix(test_data['Direction'], opt_pred)
TN, FP, FN, TP = opt_cm.ravel()

print("\nOptimal Confusion Matrix:")
print(pd.DataFrame(opt_cm,
                   index= ['Actual Down', 'Actual Up'],
                   columns= ['Pred Down',   'Pred Up']))

test_error=1-(TP+TN)/(TP+TN+FP+FN)
print(f"\nTest Error Rate: {test_error:.4f}")
print(f"Sensitivity: {TP / (TP + FN):.4f}")
print(f"Specificity: {TN / (TN + FP):.4f}")

   Cut_Point       TPR       FPR   Compare
0   0.390031  0.836066  0.883721  0.097217
1   0.608480  0.016393  0.046512  0.015631
2   0.448940  0.622951  0.581395  0.260770
3   0.410107  0.786885  0.767442  0.182997
4   0.433660  0.721311  0.651163  0.251620

Optimal row:
Cut_Point    0.457939
TPR          0.557377
FPR          0.488372
Compare      0.285170
Name: 57, dtype: float64

The cut-point for which TPR(1-FPR) is maximised: 0.4579

Optimal Confusion Matrix:
             Pred Down  Pred Up
Actual Down         22       21
Actual Up           27       34

Test Error Rate: 0.4615
Sensitivity: 0.5574
Specificity: 0.5116


We obtain the error measures from the confusion matrix.

### f. Is there any change in the test error rate if we have worked with only the two predictors Lag1 and Lag2?

In [8]:
model2= smf.logit('Direction ~ Lag1 + Lag2', data=train_data).fit()
prob= model2.predict(test_data[['Lag1', 'Lag2']])

tpr1= []
fpr1= []
measure1= []

for cutpoint in prob:
    predict_reduced=(prob >= cutpoint).astype(int)
    cm1= confusion_matrix(test_data['Direction'], predict_reduced)

    if cm1.shape==(2, 2):
        TN1,FP1,FN1,TP1 = cm1.ravel()
    else:
        TN1,FP1,FN1,TP1 = 0, 0, 0, 0

    sens1= TP1/(TP1+FN1) if (TP1 + FN1) > 0 else 0
    spec1= TN1 / (TN1 + FP1) if (TN1 + FP1) > 0 else 0
    fpr_val1 = 1 - spec1

    tpr1.append(sens1)
    fpr1.append(fpr_val1)
    measure1.append(sens1 * (1-fpr_val1))

output1 = pd.DataFrame({
    'Cut_Point': prob.values,
    'TPR': tpr1,
    'FPR': fpr1,
    'Compare': measure1
})

print(output1.head())

opt_cutpoint1= output1.loc[best_idx, 'Cut_Point']
print(output1.loc[best_idx])
print(f"\nThe cut-point for which TPR(1-FPR) is maximised: {opt_cutpoint1:.4f}")

opt_pred1= (prob >= opt_cutpoint1).astype(int)
opt_cm1= confusion_matrix(test_data['Direction'], opt_pred1)
TN1,FP1,FN1,TP1 = opt_cm1.ravel()

print("\nOptimal Confusion Matrix (Model 2):")
print(pd.DataFrame(opt_cm1,
                   index   = ['Actual Down', 'Actual Up'],
                   columns = ['Predicted Down',   'Pred Up']))

test_error1= 1-(TP1+TN1)/(TP1+TN1+FP1+FN1)
print(f"\nTest Error Rate: {test_error1:.4f}")
print(f"Sensitivity: {TP1/(TP1+FN1):.4f}")
print(f"Specificity: {TN1/(TN1+FP1):.4f}")

Optimization terminated successfully.
         Current function value: 0.683735
         Iterations 4
   Cut_Point       TPR       FPR   Compare
0   0.438614  0.967213  0.953488  0.044987
1   0.693426  0.000000  0.023256  0.000000
2   0.553916  0.573770  0.488372  0.293557
3   0.520904  0.786885  0.720930  0.219596
4   0.533847  0.737705  0.534884  0.343119
Cut_Point    0.540311
TPR          0.688525
FPR          0.534884
Compare      0.320244
Name: 57, dtype: float64

The cut-point for which TPR(1-FPR) is maximised: 0.5403

Optimal Confusion Matrix (Model 2):
             Predicted Down  Pred Up
Actual Down              20       23
Actual Up                19       42

Test Error Rate: 0.4038
Sensitivity: 0.6885
Specificity: 0.4651


### Evaluation of Model Performance

To assess the efficacy of our classification models:

- **Sensitivity (Recall):** Measures the proportion of actual "Up" days that were correctly identified by the model.
$$\text{Sensitivity} = \frac{TP}{TP + FN}$$

- **Specificity:** Measures the proportion of actual "Down" days that were correctly identified.
$$\text{Specificity} = \frac{TN}{TN + FP}$$

**Comparative Analysis:**

- **Model 1: Full Feature Set** — The full model provides a balanced predictive profile. With a sensitivity of $0.5116$ and a specificity of $0.5574$, the model demonstrates a relatively even capability in detecting both market advances and declines.

- **Model 2: Reduced Feature Set ($Lag1$ & $Lag2$)** — The streamlined model shows a shift in classification priority. While it achieves a notably higher specificity of $0.7869$, making it quite effective at identifying "Down" days, this gain comes at the expense of sensitivity, which drops to $0.4419$.